# GeoSentry — Rainfall–Terrain 24h Landslide Forecast (training)

Implements `docs/AI_MODEL_CARD.md` section 4. Read `training/README.md` and
`training/dataset_schema.md` in the repo before running this for real.

**Runtime → Change runtime type → T4 GPU (or better)** before running.

This notebook will NOT let you treat a synthetic-data run as a result. The synthetic
generator exists only to verify the pipeline runs end-to-end; its printed metrics are
watermarked `[SYNTHETIC]` everywhere, including in the saved artifacts.

In [ ]:
# ---- 0. Setup ----
!pip -q install xgboost scikit-learn pyarrow

import os, json, random, math, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    average_precision_score, roc_auc_score, brier_score_loss,
    precision_recall_curve, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

warnings.filterwarnings("ignore")

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "| AMP:", USE_AMP)
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected. Set Runtime > Change runtime type > GPU, "
          "otherwise this will be slow and defeats the point of using Colab.")

In [ ]:
# ---- 1. Mount Drive & configure paths ----
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/geosentry/dataset.parquet"  # <-- set to your real file
ARTIFACT_DIR = "/content/drive/MyDrive/geosentry/artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Minimum recall the model card requires you to state BEFORE looking at test results.
# Pick this based on what the product can tolerate (false alarms vs missed events),
# write it down, then do not change it after seeing test metrics.
MIN_VALIDATION_RECALL = 0.80

# Test = events on/after this date. Everything before is train/val pool.
TEST_CUTOFF = "2025-01-01"

## 2. Load data (falls back to synthetic pipeline-check data if `DATA_PATH` is missing)

In [ ]:
# ---- 2. Load or synthesize ----
SYNTHETIC = not os.path.exists(DATA_PATH)

def make_synthetic(n=20000, n_storms=120, n_lags=60, seed=SEED):
    """Pipeline smoke-test data only. Not representative of real rainfall/terrain/soil."""
    rng = np.random.default_rng(seed)
    storm_id = rng.integers(0, n_storms, size=n)
    issue_time = pd.Timestamp("2018-01-01") + pd.to_timedelta(
        rng.integers(0, 365 * 7, size=n), unit="D")
    slope = rng.uniform(0, 60, n)
    twi = rng.normal(8, 3, n)
    elevation = rng.uniform(50, 2000, n)
    dist_stream = rng.exponential(300, n)
    soil = rng.choice(["clay", "loam", "sand", "rock"], size=n)
    rain = rng.gamma(2.0, 8.0, size=(n, n_lags))
    rain_sum7 = rain[:, :7].sum(1)
    logit = (-6 + 0.05 * slope + 0.015 * rain_sum7 + 0.1 * twi
              - 0.0015 * dist_stream + rng.normal(0, 1, n))
    p = 1 / (1 + np.exp(-logit))
    label = (rng.uniform(0, 1, n) < p).astype(int)
    df = pd.DataFrame({
        "sample_id": [f"s{i}" for i in range(n)],
        "point_id": [f"p{i % 2000}" for i in range(n)],
        "lat": rng.uniform(8, 23, n), "lon": rng.uniform(102, 109, n),
        "issue_time": issue_time, "storm_id": [f"st{s}" for s in storm_id],
        "label": label, "slope_deg": slope, "aspect_deg": rng.uniform(0, 360, n),
        "elevation_m": elevation, "twi": twi, "dist_to_stream_m": dist_stream,
        "soil_class": soil,
    })
    for i in range(n_lags):
        df[f"rain_lag_{i+1:02d}"] = rain[:, i]
    return df

if SYNTHETIC:
    print("[SYNTHETIC] DATA_PATH not found -> using synthetic pipeline-check data. "
          "Results below are NOT real and must not be copied into the model card.")
    df = make_synthetic()
else:
    df = pd.read_parquet(DATA_PATH) if DATA_PATH.endswith(".parquet") else pd.read_csv(DATA_PATH)
    df["issue_time"] = pd.to_datetime(df["issue_time"])
    required = {"sample_id", "point_id", "issue_time", "storm_id", "label", "soil_class"}
    missing = required - set(df.columns)
    assert not missing, f"Missing required columns: {missing}"

RAIN_COLS = sorted([c for c in df.columns if c.startswith("rain_lag_")],
                    key=lambda c: int(c.split("_")[-1]))
STATIC_NUM_COLS = [c for c in ["slope_deg", "aspect_deg", "elevation_m", "twi", "dist_to_stream_m"]
                   if c in df.columns]
CAT_COLS = [c for c in ["soil_class", "land_cover"] if c in df.columns]

print(f"Rows: {len(df)} | events(label=1): {df['label'].sum()} "
      f"({df['label'].mean():.3%}) | rain lags: {len(RAIN_COLS)} | storms: {df['storm_id'].nunique()}")

## 3. Leakage-safe split: time-based test holdout + group split for train/val
No `storm_id` ever appears in more than one split. Test is strictly the latest events.

In [ ]:
# ---- 3. Split ----
test_mask = df["issue_time"] >= pd.Timestamp(TEST_CUTOFF)
pool, test_df = df[~test_mask].copy(), df[test_mask].copy()

# Storms must not straddle the cutoff: drop pool rows whose storm_id also appears in test.
leaking_storms = set(pool["storm_id"]) & set(test_df["storm_id"])
if leaking_storms:
    pool = pool[~pool["storm_id"].isin(leaking_storms)]
    print(f"Dropped {len(leaking_storms)} storms straddling the test cutoff from the train/val pool.")

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(gss.split(pool, groups=pool["storm_id"]))
train_df, val_df = pool.iloc[train_idx].copy(), pool.iloc[val_idx].copy()

assert set(train_df["storm_id"]) & set(val_df["storm_id"]) == set()
assert set(train_df["storm_id"]) & set(test_df["storm_id"]) == set()
assert set(val_df["storm_id"]) & set(test_df["storm_id"]) == set()

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s}: {len(d):6d} rows | {d['label'].sum():5d} events "
          f"| {d['storm_id'].nunique():4d} storms | "
          f"{d['issue_time'].min()} -> {d['issue_time'].max()}")

## 4. Feature prep — fit everything on `train_df` only

In [ ]:
# ---- 4. Scalers / encoders (fit on train only) ----
rain_scaler = StandardScaler().fit(train_df[RAIN_COLS].fillna(0.0))
static_scaler = StandardScaler().fit(train_df[STATIC_NUM_COLS]) if STATIC_NUM_COLS else None

cat_encoders = {}
cat_cardinality = {}
for c in CAT_COLS:
    le = LabelEncoder()
    le.fit(train_df[c].astype(str).fillna("__missing__"))
    cat_encoders[c] = le
    cat_cardinality[c] = len(le.classes_) + 1  # +1 for unseen at val/test time

def encode_cat(series, le):
    s = series.astype(str).fillna("__missing__")
    known = set(le.classes_)
    s = s.where(s.isin(known), other=le.classes_[0])  # map unseen -> a known class
    return le.transform(s)

def build_tensors(d):
    rain = rain_scaler.transform(d[RAIN_COLS].fillna(0.0)).astype(np.float32)
    rain = rain[:, :, None]  # (N, T, 1) feature dim = 1 (rainfall only); extend if more seq vars added
    static = static_scaler.transform(d[STATIC_NUM_COLS]).astype(np.float32) if STATIC_NUM_COLS else \
        np.zeros((len(d), 0), dtype=np.float32)
    cats = np.stack([encode_cat(d[c], cat_encoders[c]) for c in CAT_COLS], axis=1).astype(np.int64) \
        if CAT_COLS else np.zeros((len(d), 0), dtype=np.int64)
    y = d["label"].values.astype(np.float32)
    return rain, static, cats, y

X_train = build_tensors(train_df)
X_val = build_tensors(val_df)
X_test = build_tensors(test_df)
print("rain seq shape:", X_train[0].shape, "| static shape:", X_train[1].shape,
      "| cat shape:", X_train[2].shape)

In [ ]:
# ---- 5. Dataset / DataLoader ----
class LandslideDataset(Dataset):
    def __init__(self, tensors):
        self.rain, self.static, self.cats, self.y = tensors
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return (torch.from_numpy(self.rain[i]), torch.from_numpy(self.static[i]),
                torch.from_numpy(self.cats[i]), torch.tensor(self.y[i]))

BATCH_SIZE = 256
train_ds, val_ds, test_ds = map(LandslideDataset, [X_train, X_val, X_test])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=(DEVICE.type == "cuda"))
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=2)

## 6. Model — LSTM over rainfall sequence + static/categorical branch, fused

In [ ]:
# ---- 6. Model ----
class RainfallTerrainNet(nn.Module):
    def __init__(self, rain_feat_dim, static_dim, cat_cardinalities,
                 lstm_hidden=64, lstm_layers=1, emb_dim=4, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=rain_feat_dim, hidden_size=lstm_hidden,
                             num_layers=lstm_layers, batch_first=True,
                             dropout=dropout if lstm_layers > 1 else 0.0)
        self.embeddings = nn.ModuleList([nn.Embedding(card, emb_dim) for card in cat_cardinalities])
        fused_dim = lstm_hidden + static_dim + emb_dim * len(cat_cardinalities)
        self.head = nn.Sequential(
            nn.Linear(fused_dim, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, rain_seq, static, cats):
        _, (h_n, _) = self.lstm(rain_seq)
        rain_repr = h_n[-1]  # last layer's final hidden state: (B, hidden)
        parts = [rain_repr, static]
        for i, emb in enumerate(self.embeddings):
            parts.append(emb(cats[:, i]))
        fused = torch.cat(parts, dim=1)
        return self.head(fused).squeeze(1)  # logits

model = RainfallTerrainNet(
    rain_feat_dim=X_train[0].shape[-1],
    static_dim=X_train[1].shape[-1],
    cat_cardinalities=[cat_cardinality[c] for c in CAT_COLS],
).to(DEVICE)
print(model)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 7. Train — class-weighted loss, AMP, early stopping on validation PR-AUC

In [ ]:
# ---- 7. Training loop ----
pos = train_df["label"].sum()
neg = len(train_df) - pos
pos_weight = torch.tensor([neg / max(pos, 1)], dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

MAX_EPOCHS, PATIENCE = 60, 8
best_val_prauc, best_state, epochs_no_improve = -1.0, None, 0

@torch.no_grad()
def evaluate_logits(loader):
    model.eval()
    all_logits, all_y = [], []
    for rain, static, cats, y in loader:
        rain, static, cats = rain.to(DEVICE), static.to(DEVICE), cats.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(rain, static, cats)
        all_logits.append(logits.float().cpu().numpy())
        all_y.append(y.numpy())
    return np.concatenate(all_logits), np.concatenate(all_y)

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for rain, static, cats, y in train_loader:
        rain, static, cats, y = (rain.to(DEVICE), static.to(DEVICE),
                                  cats.to(DEVICE), y.to(DEVICE))
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(rain, static, cats)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * len(y)

    val_logits, val_y = evaluate_logits(val_loader)
    val_prauc = average_precision_score(val_y, val_logits)
    scheduler.step(val_prauc)

    improved = val_prauc > best_val_prauc
    if improved:
        best_val_prauc = val_prauc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        torch.save(best_state, os.path.join(ARTIFACT_DIR, "model_best.pt"))
    else:
        epochs_no_improve += 1

    print(f"epoch {epoch:3d} | train_loss {running_loss/len(train_ds):.4f} "
          f"| val_PR-AUC {val_prauc:.4f} {'*' if improved else ''}")

    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch} (no val PR-AUC improvement for {PATIENCE} epochs).")
        break

model.load_state_dict(best_state)
print(f"Best validation PR-AUC: {best_val_prauc:.4f}")

## 8. Baseline — XGBoost on flattened features, same split (model-card requirement)

In [ ]:
# ---- 8. Baseline ----
def flatten(d_tensors, d_df):
    rain, static, cats, y = d_tensors
    return np.concatenate([rain.squeeze(-1), static, cats.astype(np.float32)], axis=1), y

Xtr_flat, ytr = flatten(X_train, train_df)
Xval_flat, yval = flatten(X_val, val_df)
Xtest_flat, ytest = flatten(X_test, test_df)

baseline = xgb.XGBClassifier(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=float(neg / max(pos, 1)),
    eval_metric="aucpr", random_state=SEED, n_jobs=-1,
    early_stopping_rounds=30,
)
baseline.fit(Xtr_flat, ytr, eval_set=[(Xval_flat, yval)], verbose=False)
baseline_val_prauc = average_precision_score(yval, baseline.predict_proba(Xval_flat)[:, 1])
print(f"Baseline (XGBoost) validation PR-AUC: {baseline_val_prauc:.4f}")
print(f"Deep model validation PR-AUC:        {best_val_prauc:.4f}")
if best_val_prauc <= baseline_val_prauc:
    print("NOTE: deep model did not beat the baseline on validation. Per the model card, "
          "complexity must be justified by evidence -- consider shipping the baseline instead.")

## 9. Threshold selection (validation only) + calibration

In [ ]:
# ---- 9. Threshold + calibration, fit on validation, frozen before touching test ----
val_probs_raw = torch.sigmoid(torch.from_numpy(val_logits)).numpy()

# Platt scaling: fit a 1-D logistic regression of label ~ logit on validation set.
platt = LogisticRegression().fit(val_logits.reshape(-1, 1), val_y)
def calibrate(logits):
    return platt.predict_proba(logits.reshape(-1, 1))[:, 1]
val_probs = calibrate(val_logits)

precisions, recalls, thresholds = precision_recall_curve(val_y, val_probs)
feasible = recalls[:-1] >= MIN_VALIDATION_RECALL
if feasible.any():
    best_i = np.argmax(precisions[:-1] * feasible)  # highest precision among feasible
    chosen_threshold = float(thresholds[best_i])
else:
    chosen_threshold = float(thresholds[np.argmax(recalls[:-1])])
    print(f"WARNING: no threshold reaches MIN_VALIDATION_RECALL={MIN_VALIDATION_RECALL}; "
          f"using the highest-recall threshold found ({recalls.max():.3f}). "
          "Report this gap -- do not quietly lower the target.")

print(f"Frozen threshold (chosen on validation ONLY): {chosen_threshold:.4f}")
print(f"At this threshold on validation: "
      f"recall={recalls[best_i]:.3f}, precision={precisions[best_i]:.3f}")

## 10. Final test evaluation — run exactly once

In [ ]:
# ---- 10. Test evaluation (single run; do not loop back to change the threshold) ----
test_logits, test_y = evaluate_logits(test_loader)
test_probs = calibrate(test_logits)
test_pred = (test_probs >= chosen_threshold).astype(int)

pr_auc = average_precision_score(test_y, test_probs)
roc_auc = roc_auc_score(test_y, test_probs) if len(np.unique(test_y)) > 1 else float("nan")
brier = brier_score_loss(test_y, test_probs)
tn, fp, fn, tp = confusion_matrix(test_y, test_pred, labels=[0, 1]).ravel()
recall = tp / max(tp + fn, 1)
precision = tp / max(tp + fp, 1)
n_time_units = test_df["point_id"].nunique() * max(
    (test_df["issue_time"].max() - test_df["issue_time"].min()).days, 1)
false_alarms_per_unit_time = fp / max(n_time_units, 1)

tag = "[SYNTHETIC] " if SYNTHETIC else ""
print(f"{tag}=== TEST RESULTS (window-level, not event-level) ===")
print(f"{tag}samples: {len(test_y)} | positive windows: {int(test_y.sum())} "
      f"| distinct storms in test: {test_df['storm_id'].nunique()}")
print(f"{tag}PR-AUC: {pr_auc:.4f} | ROC-AUC: {roc_auc:.4f} | Brier: {brier:.4f}")
print(f"{tag}At threshold {chosen_threshold:.4f}: recall={recall:.3f}, precision={precision:.3f}")
print(f"{tag}Confusion matrix: TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"{tag}False alarms per point-day in test window: {false_alarms_per_unit_time:.5f}")
print(f"{tag}Baseline XGBoost test PR-AUC: "
      f"{average_precision_score(ytest, baseline.predict_proba(Xtest_flat)[:, 1]):.4f}")
print()
print("REMINDER: these are WINDOW-level metrics. Event-level recall (did we warn before "
      "each real event, regardless of how many windows it spans) must be computed "
      "separately by aggregating over storm_id/point_id before reporting to the model card.")

## 11. Save artifacts + a model-card-ready results block

In [ ]:
# ---- 11. Save ----
import pickle, hashlib

run_id = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
run_dir = os.path.join(ARTIFACT_DIR, run_id)
os.makedirs(run_dir, exist_ok=True)

torch.save(model.state_dict(), os.path.join(run_dir, "model.pt"))
with open(os.path.join(run_dir, "scalers.pkl"), "wb") as f:
    pickle.dump({"rain_scaler": rain_scaler, "static_scaler": static_scaler,
                 "cat_encoders": cat_encoders}, f)
with open(os.path.join(run_dir, "platt_calibrator.pkl"), "wb") as f:
    pickle.dump(platt, f)

data_fingerprint = hashlib.sha256(pd.util.hash_pandas_object(df, index=True).values).hexdigest()[:16]
metadata = {
    "run_id": run_id, "synthetic": SYNTHETIC, "seed": SEED,
    "data_fingerprint": data_fingerprint, "data_path": DATA_PATH,
    "test_cutoff": TEST_CUTOFF, "min_validation_recall_target": MIN_VALIDATION_RECALL,
    "rain_lag_columns": RAIN_COLS, "static_columns": STATIC_NUM_COLS, "cat_columns": CAT_COLS,
    "chosen_threshold": chosen_threshold,
    "val_pr_auc": float(best_val_prauc), "baseline_val_pr_auc": float(baseline_val_prauc),
    "test_pr_auc": float(pr_auc), "test_roc_auc": float(roc_auc), "test_brier": float(brier),
    "test_recall": float(recall), "test_precision": float(precision),
    "test_confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
    "n_storms_train": int(train_df["storm_id"].nunique()),
    "n_storms_val": int(val_df["storm_id"].nunique()),
    "n_storms_test": int(test_df["storm_id"].nunique()),
}
with open(os.path.join(run_dir, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"Saved artifacts to {run_dir}")
print(json.dumps(metadata, indent=2, default=str))
if SYNTHETIC:
    print("\n[SYNTHETIC RUN] Do not copy these numbers into docs/AI_MODEL_CARD.md. "
          "Re-run with a real DATA_PATH first.")

## Next steps

1. Build the real dataset per `dataset_schema.md` (rainfall history + DEM terrain +
   soil, joined to a labeled landslide inventory with `storm_id`/time grouping).
2. Point `DATA_PATH` at it and re-run top to bottom.
3. Compute event-level recall (not just window-level) by aggregating predictions per
   `storm_id`/`point_id` before declaring a result.
4. Copy the real `metadata.json` numbers into `docs/AI_MODEL_CARD.md` section 5,
   replacing the "Not measured" rows, and update section 5's status line.
5. Only then wire a trained artifact into `services/ai-service/app/services/inference.py`
   (which currently still contains the placeholder slope/NDVI formula noted in the
   model card).